In [2]:
import time
import matplotlib.pyplot as plt
import numpy as np
from typing import Callable, Tuple

def timeit(f:Callable, *args, n:int=1)->float:
    """Times the function `f` called with arguments `args` `n` times returns the average time in seconds."""
    start_time = time.perf_counter()
    for _ in range(n):
        f(*args)
    end_time = time.perf_counter()
    return (end_time - start_time) / n




> **Rod Cutting Problem — Dynamic Programming**
> This notebook walks through the classic rod-cutting optimization problem, building up from a naive
> exponential recursive solution to a fully optimized iterative dynamic programming algorithm capable of
> handling rods of length 100,000+ in linear time. See the [README](README.md) for a full overview.


# Cutting Rods

You have a rod (stick) of length $n$ of some ressource (say copper) you want to sell.

You can cut it in pieces of integer lengths $l$ that sell at different prices $p_l$.

The order of lengths does not matter, we can write your cutting choice as a partition of $n$: 

$n = \sum_{i} l_i$. e.g. a rod of length 3 can be cut as follows: $3 = 1+ 1 + 1; 3 = 1 + 2;$ and $3 = 3$.

For now assume cutting the rod costs you nothing. 

Given a list of prices $p=[0, p_1, p_2,..., p_n]$.
- What is the maximum value you can obtain for your rod of size $n$? 
- What is the partition associated with the obtained value? 


### Example:
`p = [0,2,3,7,10]`

What is the maximum value you can get rods of different lenghts?

if

- $n = 1 \to v = 2$
- $n = 2 \to v = 4\; (n =1+1)$
- $n = 3 \to v = 7  \;(n =3)$
- $n = 4 \to v = 10\;(n=4)$
- $n = 5 \to v = 12\;(n=1+4)$
- $n = 6 \to v = 14\;(n=1+1+4)$
- $n = 7 \to v = 17\;(n=3+4)$
- $n = 8 \to v = 20\;(n=4+4)$

Recursively calculate the maximum value obtained for a rod of length $n$ given the prices list for each integer length.

Recurse over each cut. Each cut generates smaller problems.

**Use constant extra space. No extra lists**

Additional hint: 

- Suppose you know the best cut position: at length `i`, and sell that piece for the price `p[i]`. what is the relationship between the value obtained for a rod of length `n` and the value obtained for the remaining piece of length `n-i`? 
- How can you get the best cut position `i`?

We start with a simple example where only lengths smaller than 4 can be sold.

In [3]:
p = [0,2,3,7,10]

In [4]:
def rod_cut(n:int, p:list)->int:
    """ Returns the maximum value obtainable by cutting a rod of length n
        given the prices p for each length. Does not need to keep track of cuts positions.
    Args:
        n (int): length of the rod
        p (list): prices for each length. Always >= 0 and p[0] = 0.
    Returns:
        int: maximum obtainable value
    """
    if n == 0:
        return 0
    max_val = -1 # value cannot be negative, so this is safe
    
    for i in range(1,len(p)): # TODO: Careful with the lengths of p and the rod
        if i>n:
               break
        if max_val<p[i]+rod_cut(n-i,p):
           max_val=p[i]+rod_cut(n-i,p)             
    return max_val
    



In [5]:
# Test the function

for n in range(0,10):
    print(f'Maximum obtainable value for rod of length {n}: {rod_cut(n, p)}')



Maximum obtainable value for rod of length 0: 0
Maximum obtainable value for rod of length 1: 2
Maximum obtainable value for rod of length 2: 4
Maximum obtainable value for rod of length 3: 7
Maximum obtainable value for rod of length 4: 10
Maximum obtainable value for rod of length 5: 12
Maximum obtainable value for rod of length 6: 14
Maximum obtainable value for rod of length 7: 17
Maximum obtainable value for rod of length 8: 20
Maximum obtainable value for rod of length 9: 22


Now you have the value but we also want the partition that generated it.

Rewrite the algorithm to also return the cuts that generated the maximum value.

Jointly build the value and the cuts list.

In [6]:
def rod_cut_with_cuts(n:int, p:list)->tuple:
    """ Returns the maximum value obtainable by cutting a rod of length n
        given the prices p for each length, along with the cuts made.
    Args:
        n (int): length of the rod
        p (list): prices for each length. Always >= 0 and p[0] = 0.
    Returns:
        tuple: maximum obtainable value and list of cuts made
    """
    if n == 0:
        return 0, []
    max_val = -1
    best_cuts = []
    for i in range(1,len(p)): # TODO: Careful with the lengths of p and the rod
        if i>n:
               break
        if max_val<p[i]+rod_cut_with_cuts(n-i,p)[0]:
           max_val=p[i]+rod_cut_with_cuts(n-i,p)[0]
           m=i
    best_cuts.append(m)
    best_cuts.append(rod_cut_with_cuts(n-m,p)[1])
    return max_val, best_cuts


In [7]:

# Test the function with cuts   
for n in range(0,10):
    max_value, cuts = rod_cut_with_cuts(n, p)
    print(f'Maximum obtainable value for rod of length {n}: {max_value} with cuts: {cuts}')
    

Maximum obtainable value for rod of length 0: 0 with cuts: []
Maximum obtainable value for rod of length 1: 2 with cuts: [1, []]
Maximum obtainable value for rod of length 2: 4 with cuts: [1, [1, []]]
Maximum obtainable value for rod of length 3: 7 with cuts: [3, []]
Maximum obtainable value for rod of length 4: 10 with cuts: [4, []]
Maximum obtainable value for rod of length 5: 12 with cuts: [1, [4, []]]
Maximum obtainable value for rod of length 6: 14 with cuts: [1, [1, [4, []]]]
Maximum obtainable value for rod of length 7: 17 with cuts: [3, [4, []]]
Maximum obtainable value for rod of length 8: 20 with cuts: [4, [4, []]]
Maximum obtainable value for rod of length 9: 22 with cuts: [1, [4, [4, []]]]


### Complexity


In [8]:
# Time it

lengths = list(range(1,27))
times_fixed_length = [timeit(rod_cut, n, p, n=1) for n in lengths]
plt.plot(lengths, times_fixed_length, label='Fixed len(p)')
plt.xlabel('Rod Length (n)')
plt.ylabel('Time (s)')
plt.title(f'Rod Cutting Time Complexity - len(p)={len(p)}')

KeyboardInterrupt: 

The time complexity is exponential, can you see why?

This is despite only allowing `len(p)` lengths to be sold.

When `len(p) == n`, the complexity is $\Theta(2^n)$. **Prove it as an exercise later**.

Let's have a look at the case where every length can be sold.

In [ ]:
times_all_lengths = []
for n in lengths:
    p_ = np.random.randint(1,n+1, size=n+1).tolist()
    p_.sort()
    p_[0] = 0
    times_all_lengths.append(timeit(rod_cut, n, p_, n=1))

KeyboardInterrupt: 

In [ ]:
plt.plot(lengths, times_fixed_length, label=f'Fixed len(p)={len(p)}')
plt.plot(lengths, times_all_lengths, label='len(p)=n')
plt.xlabel('Rod Length (n)')
plt.ylabel('Time (s)')
# plt.yscale('log')
plt.title('Rod Cutting Time Complexity')
plt.legend()
plt.show()

# Can we do better?
To build your recursive algorithm, you most likely already thought about the subproblems of the initial problem: To optimally cut a rod of length `n`, you need to optimally cut rods of lengths `l` for `l<n`.

How many distinct subproblems are there?

There are only `n` distinct subproblems, one for each length from `1` to `n`.

Each subproblem has the same solution for a fixed prices list `p`.

The recursive algorithm is not efficient because it recomputes the same subproblems multiple times.

Use **memoization** to store the solutions of each subproblem you encounter and reuse them when needed. (use a memo table of size that holds the solutions for lengths from `0` to `n`)

This is actually the trick we used to make Fibonacci run in linear time in the first lab.

In [9]:
def rod_cut_memo(n:int, p:list, memo:dict={})->int:
    """ Returns the maximum value obtainable by cutting a rod of length n
        given the prices p for each length, using memoization.
    Args:
        n (int): length of the rod
        p (list): prices for each length. Always >= 0 and p[0] = 0.
    Returns:
        int: maximum obtainable value
    """
    if n in memo:
        return memo[n]
    if n == 0:
        return 0
    max_val = -1
    for i in range(1,len(p)):
        if i>n:
               break
        if max_val<p[i]+rod_cut_memo(n-i,p,memo):
           max_val=p[i]+rod_cut_memo(n-i,p,memo)
           m=i
          
    memo[n]=max_val
    
    return max_val

In [10]:
# Verify you get the same results with memoization
for n in range(0,10):
    print(f'Maximum obtainable value for rod of length {n} with memoization: {rod_cut_memo(n, p)}')

Maximum obtainable value for rod of length 0 with memoization: 0
Maximum obtainable value for rod of length 1 with memoization: 2
Maximum obtainable value for rod of length 2 with memoization: 4
Maximum obtainable value for rod of length 3 with memoization: 7
Maximum obtainable value for rod of length 4 with memoization: 10
Maximum obtainable value for rod of length 5 with memoization: 12
Maximum obtainable value for rod of length 6 with memoization: 14
Maximum obtainable value for rod of length 7 with memoization: 17
Maximum obtainable value for rod of length 8 with memoization: 20
Maximum obtainable value for rod of length 9 with memoization: 22


Is it more efficient?

Let's try a large `n` that was previously unsolvable.

In [11]:

n = 100 # Previously unsolvable.
ps = np.random.randint(1, n, size=n+1).tolist()
ps[0] = 0
ps.sort()
print(f'Maximum obtainable value for rod of length {n} with memoization: {rod_cut_memo(n, ps)}')

Maximum obtainable value for rod of length 100 with memoization: 342


Cool, but how do we keep track of the cuts now?

Modify your algorithm so that it also keeps track of the last cut made for each subproblem. No need to store all cuts, just the last one.

In [12]:
def rod_cut_memo_cuts(n:int, p:list, memo:dict)->int:
    """ Returns the maximum value obtainable by cutting a rod of length n
        given the prices p for each length, using memoization. 
        In the memo, we also store the last cut made to reconstruct the cuts later.
    Args:
        n (int): length of the rod
        p (list): prices for each length. Always >= 0 and p[0] = 0.
        memo (dict): memoization dictionary to store values and last cuts. 
                     Pass in an empty dict initially to have access to it after the function returns. (required)
                     memo[i] = (max_value, last_cut)
    Returns:
        int: maximum obtainable value

    Modifies memo in place 
    """
    if n in memo:
        return memo[n][0]
    if n == 0:
        return 0
    max_val = -1
    for i in range(1,len(p)):
        if i>n:
               break
        if max_val<p[i]+rod_cut_memo_cuts(n-i,p,memo):
           max_val=p[i]+rod_cut_memo_cuts(n-i,p,memo)
           m=i
    memo[n]=(max_val,m) 
    return max_val

# Verify you get the same results with memoization and cuts tracking
for n in range(0,10):
    max_value = rod_cut_memo_cuts(n, p, {})
    print(f'Maximum obtainable value for rod of length {n} with memoization and cuts tracking: {max_value}')


Maximum obtainable value for rod of length 0 with memoization and cuts tracking: 0
Maximum obtainable value for rod of length 1 with memoization and cuts tracking: 2
Maximum obtainable value for rod of length 2 with memoization and cuts tracking: 4
Maximum obtainable value for rod of length 3 with memoization and cuts tracking: 7
Maximum obtainable value for rod of length 4 with memoization and cuts tracking: 10
Maximum obtainable value for rod of length 5 with memoization and cuts tracking: 12
Maximum obtainable value for rod of length 6 with memoization and cuts tracking: 14
Maximum obtainable value for rod of length 7 with memoization and cuts tracking: 17
Maximum obtainable value for rod of length 8 with memoization and cuts tracking: 20
Maximum obtainable value for rod of length 9 with memoization and cuts tracking: 22


In [13]:
# Have a look at the cuts made for a specific length
n = 9
memo = {}
max_value = rod_cut_memo_cuts(n, p, memo)
print(memo)

{1: (2, 1), 2: (4, 1), 3: (7, 3), 4: (10, 4), 5: (12, 1), 6: (14, 1), 7: (17, 3), 8: (20, 4), 9: (22, 1)}


Walk your way back up in the memo to reconstruct the cuts.

In the example, for n=9, memo[9] is telling me the last cut was 1. I sell 1 and a rod of length 8 remains.

Then memo[8] tells me the last cut was 4. I sell 4 and a rod of length 4 remains.

Then memo[4] tells me the last cut was 4. I sell 4 and a rod of length 0 remains. Done. The lengths are 1,4,4.

In [14]:
def get_lengths_from_memo(memo:dict)->list:
    """ Reconstructs the cuts made to obtain the maximum value for a rod of length n
        using the memoization dictionary.
    Args:
        n (int): length of the rod
        memo (dict): memoization dictionary with last cuts information in the form {length: (max_value, last_cut)}.
    Returns:
        list: list of cuts made to obtain the maximum value
    """
    n = max(memo.keys())
    cuts = []
    while n>0:
       last_cut=memo[n][1]
       cuts.append(last_cut)
       n-=last_cut
    return cuts


In [15]:
# Check the cuts made
lengths = get_lengths_from_memo(memo)
print(f'Cuts made for rod of length {max(memo.keys())}: {lengths}')

Cuts made for rod of length 9: [1, 4, 4]


The memoized cut rod algorithm runs in linear time.

We should be able to evaluate rods of length up to 10^5 in a reasonable time.

In [ ]:
n = 10000
rod_cut_memo_cuts(n, p, memo)

: 

Then what's wrong?

The algorithm is recursive and Python has a recursion limit (default 1000).

You could force Python to increase its recursion limit but a better solution is to rewrite the algorithm in an iterative manner.

# Iterative Rod Cut with Memoization and Cuts Tracking

In the recursive memoized version of the rod cut algorithm, we fill the memo table on demand, meaning that when we need the solution for a subproblem of length `k`, we check if it's already in the memo. If not, we compute it recursively.

The most important in this iterative rewrite is to fill the memo table iteratively from the base case up to the desired length `n` in the order of dependency, in this case from length `0` to length `n`.

In [22]:
def rod_cut_memo_iterative(n:int, p:list)->Tuple[int, list]:
    """ Returns the maximum value obtainable by cutting a rod of length n
        given the prices p for each length, using an iterative bottom-up approach.
    Args:
        n (int): length of the rod
        p (list): prices for each length. Always >= 0 and p[0] = 0.
    
    Returns:
        tuple: maximum obtainable value and list of cuts made
    """
    memo = {0: (0,0)}  # length: (max_value, last_cut)
    # [TODO] Loop over every subproblem in "topological order" (in this case increasing lengths) 
    # Solve them and store the results in memo
    for i in range(1,n+1):
        m=0
        max_value=-1
        for j in range(1,min(i+1,len(p))):
            if max_value<p[j]+memo[i-j][0]:
                max_value=p[j]+memo[i-j][0]
                m=j
        memo[i]=max_value,m
    lengths = get_lengths_from_memo(memo)
    return memo[n][0],lengths
        

In [23]:
# Verify you get the same results with the iterative approach
for n in range(0,10):
    max_value, cuts = rod_cut_memo_iterative(n, p)
    print(f'Maximum obtainable value for rod of length {n} with iterative approach: {max_value} with cuts: {cuts}')
    

Maximum obtainable value for rod of length 0 with iterative approach: 0 with cuts: []
Maximum obtainable value for rod of length 1 with iterative approach: 2 with cuts: [1]
Maximum obtainable value for rod of length 2 with iterative approach: 4 with cuts: [1, 1]
Maximum obtainable value for rod of length 3 with iterative approach: 7 with cuts: [3]
Maximum obtainable value for rod of length 4 with iterative approach: 10 with cuts: [4]
Maximum obtainable value for rod of length 5 with iterative approach: 12 with cuts: [1, 4]
Maximum obtainable value for rod of length 6 with iterative approach: 14 with cuts: [1, 1, 4]
Maximum obtainable value for rod of length 7 with iterative approach: 17 with cuts: [3, 4]
Maximum obtainable value for rod of length 8 with iterative approach: 20 with cuts: [4, 4]
Maximum obtainable value for rod of length 9 with iterative approach: 22 with cuts: [1, 4, 4]


Now we can run it on huge rods.

In [24]:
n = 100003
rod_cut_memo_iterative(n, p)

(250007,
 [3,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
  4,
 

# Welcome to Dynamic Programming!

A similar problem to think about is the **coin change problem**:

Given a list of coin denominations and a target amount, find the minimum number of coins needed to make that amount.

## Summary

| Approach | Time Complexity | Space Complexity | Max practical `n` |
|---|---|---|---|
| Naive recursion | $\Theta(2^n)$ | $O(n)$ (call stack) | ~25 |
| Memoized (top-down) | $O(n \cdot \lvert p \rvert)$ | $O(n)$ | ~900 (recursion limit) |
| Iterative (bottom-up) | $O(n \cdot \lvert p \rvert)$ | $O(n)$ | 100,000+ |

The exponential blow-up of the naive approach comes from repeatedly solving the same overlapping subproblems.
Memoization fixes this by caching results, and the iterative bottom-up version removes the Python recursion
limit entirely by solving subproblems in increasing order of rod length.

This pattern — identify overlapping subproblems, cache or tabulate, then convert to bottom-up — generalizes to
many other DP problems (coin change, longest common subsequence, knapsack, edit distance, and the optimal BST
problem in [this related project](https://github.com/YOUR_USERNAME/optimal-bst-dp)).